# MS4: Hyperbolic Embeddings for Hierarchical Style Representation

**Canvas Project #150**  
**Group Members:** Peter Flo, Luca Grossmann, Valerie Wang  
**COMPSCI 2090B — Spring 2026**

This notebook is the main deliverable for MS4. It walks through the final modeling pipeline end-to-end: scaling our MS3 baseline into a multi-axis grid search, adding a hierarchy-aware loss, and probing the result against alternative ground-truth trees. The accompanying report (`report/main.pdf`) carries the prose argument; this notebook carries the executable evidence.

## How to read this notebook

- Sections 0–3 are setup, data recap, and methods.
- Section 4 reproduces a single reference run end-to-end so the grader can verify the pipeline runs cleanly.
- Sections 5–7 load `data/runs/sweep.csv` (produced by `python scripts/sweep.py --phase {1,2,3}`) and present the headline results.
- Section 8 is qualitative analysis (Poincaré-disk plots, nearest-neighbor grids).
- Section 9 is the final summary table.

Helper code lives in `scripts/`; this notebook imports from it. A precomputed `data/runs/sweep.csv` is checked in alongside this notebook so the analysis cells run without re-doing the full sweep.

## 0. Setup

In [ ]:
# Uncomment if running on Colab or a fresh machine.
# %pip install --quiet -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Add repo scripts/ to import path so we can call our pipeline directly.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from hierarchy import STYLE_HIERARCHY, distance_matrix, load_style_classes

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

DATA_DIR = REPO_ROOT / "data"
RUNS_DIR = DATA_DIR / "runs"

# The notebook prefers the live sweep CSV produced by `scripts/sweep.py` if
# available, otherwise falls back to a checked-in snapshot in notebooks/.
# This lets a grader run the analysis cells without reproducing the sweep.
LIVE_SWEEP_CSV = RUNS_DIR / "sweep.csv"
SNAPSHOT_SWEEP_CSV = REPO_ROOT / "notebooks" / "sweep_results.csv"
SWEEP_CSV = LIVE_SWEEP_CSV if LIVE_SWEEP_CSV.exists() else SNAPSHOT_SWEEP_CSV

print("repo root:", REPO_ROOT)
print("sweep csv:", SWEEP_CSV.relative_to(REPO_ROOT) if SWEEP_CSV.exists() else "(none yet)")

## 1. Background and Motivation

*See the report for a full version. Brief recap below.*

Artistic style is hierarchical: broad traditions branch into fine-grained movements. Standard image embeddings live in Euclidean space, where ball volume grows polynomially with radius. Hyperbolic geometry, where volume grows exponentially, has been argued to be a more natural home for tree-like data (Nickel & Kiela, 2017; Ganea et al., 2018; Khrulkov et al., 2020).

**Refined research question (from MS3):** *Do Poincaré-ball embeddings trained on frozen CLIP features better recover the WikiArt style hierarchy — measured by tree distortion, dendrogram agreement, and sibling/cousin retrieval — compared to Euclidean embeddings of matched dimensionality and training budget?*

**MS3 finding:** at d=8, c=1, 10 epochs, the Euclidean baseline wins on most global metrics; the hyperbolic model only wins on local sibling/cousin retrieval. MS4 asks whether that gap closes when we (a) sweep dimensionality and curvature, (b) add a hierarchy-aware regularizer that uses the tree at training time, and (c) test sensitivity to the choice of ground-truth tree.

## 2. Data and EDA Recap

We use the WikiArt Refined dataset (Tan et al., 2019) — ~81k digitized paintings labelled with 27 style classes. The 70/30 train/val split is supplied with the data. Two key findings from MS2 drive modeling decisions:

1. **Severe class imbalance.** A handful of styles (Impressionism, Realism) dominate; several substyles (e.g. Action_painting, Synthetic_Cubism) have only a few hundred examples. This pushes us toward balanced-accuracy reporting and prototype classifiers rather than a softmax tied to class frequency.
2. **Style hierarchy is not unique.** WikiArt does not ship one. We hand-built a lineage tree based on standard art-history references (see `scripts/hierarchy.py`). Because that choice is not unique, MS4 includes a sensitivity check against two alternative trees — `chronological` (era-based) and `flat` (no hierarchy at all, used as a null).

The cell below visualizes the three trees as distance matrices and reports their off-diagonal mean / max.

In [ ]:
style_names = load_style_classes()
trees = {name: distance_matrix(style_names, hierarchy_name=name)
         for name in ["default", "chronological", "flat"]}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (name, T) in zip(axes, trees.items()):
    off = T[T > 0]
    sns.heatmap(T, ax=ax, cmap="mako", cbar=False, square=True, xticklabels=False, yticklabels=False)
    ax.set_title(f"{name}\nmean={off.mean():.2f}, max={T.max()}")
plt.suptitle("Tree distance matrices over the 27 WikiArt styles")
plt.tight_layout()
plt.show()

## 3. Methods

**Architecture.** A two-layer MLP maps frozen CLIP ViT-B/16 features (512-d) to a $d$-dimensional embedding. A geometry-specific head produces the final embedding: identity for Euclidean, exponential map at the origin for the Poincaré ball. A prototype classifier with one learnable point per style produces logits as $-\text{dist}(z, p_k)$, where the distance is Euclidean or Poincaré respectively.

**Hyperbolic distance** on the Poincaré ball with curvature $c$:
$$d^c(x, y) = \frac{1}{\sqrt{c}} \operatorname{arcosh}\left( 1 + \frac{2c\,\|x - y\|^2}{(1 - c\,\|x\|^2)(1 - c\,\|y\|^2)} \right).$$

**Optimization.** Adam for the head; `RiemannianAdam` (`geoopt`) for the hyperbolic prototypes so the manifold constraint is enforced on every step.

**Hierarchy-aware loss (new in MS4).** We add a regularizer that pushes the prototype-prototype distance matrix toward the tree distance matrix:
$$\mathcal{L} = \mathcal{L}_{CE} + \lambda \cdot \frac{1}{|\mathcal{P}|} \sum_{(i,j) \in \mathcal{P}} \left( \frac{d(p_i, p_j)}{\bar{d}} - \frac{T_{ij}}{\bar{T}} \right)^2$$
where $\mathcal{P}$ is the set of upper-triangular pairs and the mean-normalization makes the loss scale-invariant (so it doesn't fight the classification term for absolute embedding magnitude). $\lambda = 0$ recovers the MS3 baseline.

Code lives in [`scripts/models.py`](../scripts/models.py), [`scripts/train.py`](../scripts/train.py) (see `tree_regularizer`), and [`scripts/eval.py`](../scripts/eval.py).

## 4. Reproduce a Single Reference Run

This cell trains and evaluates a single Euclidean and a single hyperbolic model at the MS3 baseline configuration (d=8, c=1, 5 epochs). It exists so a grader can verify the pipeline is wired correctly without re-running the full sweep (which trains ~150 configs and takes hours).

In [ ]:
from train import train_euclidean, train_hyperbolic
from eval import run_evaluation

REF_DIR = RUNS_DIR / "ms4_reference"

eu = train_euclidean(
    dim=8, epochs=5, batch_size=4096,
    run_dir=REF_DIR / "euclidean", seed=0,
)
hy = train_hyperbolic(
    dim=8, epochs=5, batch_size=4096, curvature=1.0,
    run_dir=REF_DIR / "hyperbolic", seed=0,
)

eu_metrics = run_evaluation(
    ckpt_path=Path(eu["ckpt_path"]), split="val", batch_size=4096,
    output_dir=REF_DIR / "euclidean" / "eval_val",
)
hy_metrics = run_evaluation(
    ckpt_path=Path(hy["ckpt_path"]), split="val", batch_size=4096,
    output_dir=REF_DIR / "hyperbolic" / "eval_val",
)

compare_keys = ["top1_accuracy", "top5_accuracy", "balanced_accuracy",
                "class_center_tree_spearman", "tree_distortion_average",
                "dendrogram_cluster_f1", "knn_siblings_recall_at_5",
                "knn_cousins_recall_at_5"]
ref_table = pd.DataFrame({
    "Euclidean (d=8)": [eu_metrics[k] for k in compare_keys],
    "Hyperbolic (d=8, c=1)": [hy_metrics[k] for k in compare_keys],
}, index=compare_keys)
ref_table

## 5. Phase 1 — Scaling Sweep

**Question:** does Hyperbolic catch up if we give it more dimensions or a different curvature?

**Sweep:** geometry $\in$ {euclidean, hyperbolic}; $d \in$ {2, 4, 8, 16, 32, 64}; $c \in$ {0.1, 0.3, 1.0, 3.0} (hyperbolic only); 30 epochs; 3 seeds. ~150 configs total. Reproduce with `python scripts/sweep.py --phase 1`.

In [ ]:
if not SWEEP_CSV.exists():
    print("sweep.csv not found — run `python scripts/sweep.py --phase 1` first.")
    sweep = None
else:
    sweep = pd.read_csv(SWEEP_CSV)
    print("sweep rows:", len(sweep))
    print("phase 1 rows (tree_loss=0, default tree):",
          ((sweep["tree_loss_weight"] == 0) & (sweep["tree_hierarchy"] == "default")).sum())
    sweep.head()

In [ ]:
# Phase 1 plot: top-1, tree Spearman, sibling recall@5 as a function of d, by geometry.
# For hyperbolic, we take the BEST curvature at each d (per-d, per-seed) before averaging.
if sweep is not None:
    p1 = sweep[(sweep["tree_loss_weight"] == 0) & (sweep["tree_hierarchy"] == "default")].copy()

    metrics = [("top1_accuracy", "Top-1 accuracy"),
               ("class_center_tree_spearman", "Class-center / tree Spearman"),
               ("knn_siblings_recall_at_5", "Sibling recall@5")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (col, label) in zip(axes, metrics):
        for geo, marker in [("euclidean", "o"), ("hyperbolic", "s")]:
            sub = p1[p1["geometry"] == geo]
            if geo == "hyperbolic":
                # best curvature per (dim, seed), then mean over seeds
                best = sub.loc[sub.groupby(["dim", "seed"])[col].idxmax()]
                summary = best.groupby("dim")[col].agg(["mean", "std"]).reset_index()
            else:
                summary = sub.groupby("dim")[col].agg(["mean", "std"]).reset_index()
            ax.errorbar(summary["dim"], summary["mean"], yerr=summary["std"],
                        marker=marker, label=geo, capsize=3)
        ax.set_xscale("log", base=2)
        ax.set_xlabel("embedding dim d")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend()
    plt.suptitle("Phase 1 — scaling with d (hyperbolic uses best c per d)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Curvature sweep at the most informative d (we'll pick whichever d shows the
# largest hyperbolic improvement post-sweep). Default to d=8 to compare with MS3.
if sweep is not None:
    p1c = sweep[(sweep["tree_loss_weight"] == 0)
                & (sweep["tree_hierarchy"] == "default")
                & (sweep["geometry"] == "hyperbolic")
                & (sweep["dim"] == 8)].copy()
    if len(p1c) > 0:
        summary = p1c.groupby("curvature")[["top1_accuracy", "class_center_tree_spearman",
                                            "knn_siblings_recall_at_5"]].agg(["mean", "std"])
        print("Hyperbolic d=8: metric mean±std vs curvature")
        print(summary)
    else:
        print("No phase-1 hyperbolic d=8 rows yet.")

### Phase 1 finding: scaling does not break the local–global trade-off

Three things stand out across 90 seed-replicated runs.

**(1) Top-1 accuracy: Euclidean wins everywhere except $d=2$.** Both geometries saturate by $d \approx 16$ (Euclidean at 65.4–65.9%, hyperbolic at 60.2–60.6%). At $d=2$ the two are statistically tied (53.2% vs 53.1%, seed std $\le$ 0.6 pp). For $d \in \{4, 8, 16, 32, 64\}$ Euclidean leads by 4–6 percentage points and the gap is stable, not closing.

**(2) Local hierarchy structure: hyperbolic wins everywhere except $d=2$, and the gap *widens* with $d$.** Sibling recall@5 is essentially tied at $d=2$ (0.221 vs 0.218) but Euclidean drops to 0.142 by $d=64$ while hyperbolic only drops to 0.188 — a 5-point absolute lead in the high-$d$ regime where Euclidean has the most embedding capacity to use. This is the MS3 observation amplified.

**(3) Within hyperbolic, curvature trades top-1 for hierarchy.** At $d=8$, $c=0.3$ gives the best top-1 (59.8%) but worse sibling recall (0.192); $c=3.0$ gives the best sibling recall (0.230) but worst top-1 (57.4%). The Poincaré ball is "more boundary-like" at higher curvature and that helps local structure at a small classification cost.

**Implication for Phase 2.** Pure scaling cannot close the global gap. If a hierarchy-aware regularizer can pull the hyperbolic global metrics up without sacrificing the local advantage, that's the headline. Phase 2 sweeps $\lambda$ at $d=8$ to test this directly.

## 6. Phase 2 — Hierarchy-Aware Loss

**Question:** if we give both models the tree at training time, does the hyperbolic model close the gap?

**Sweep:** $\lambda \in$ {0, 0.1, 0.3, 1.0, 3.0}; $d \in$ {8, 16}; $c=1$; 3 seeds. Reproduce with `python scripts/sweep.py --phase 2`.

In [ ]:
if sweep is not None:
    p2 = sweep[(sweep["tree_hierarchy"] == "default")
               & (sweep["tree_loss_weight"].isin([0.0, 0.1, 0.3, 1.0, 3.0]))
               & (sweep["dim"].isin([8, 16]))].copy()
    print("phase 2 rows:", len(p2))

    if len(p2) > 0:
        metric_cols = ["top1_accuracy", "class_center_tree_spearman",
                       "tree_distortion_average", "knn_siblings_recall_at_5"]
        fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
        for ax, col in zip(axes, metric_cols):
            for geo, marker in [("euclidean", "o"), ("hyperbolic", "s")]:
                sub = p2[(p2["geometry"] == geo) & (p2["dim"] == 8)]
                summary = sub.groupby("tree_loss_weight")[col].agg(["mean", "std"]).reset_index()
                ax.errorbar(summary["tree_loss_weight"], summary["mean"], yerr=summary["std"],
                            marker=marker, label=geo, capsize=3)
            ax.set_xscale("symlog", linthresh=0.05)
            ax.set_xlabel("λ (tree loss weight)")
            ax.set_ylabel(col)
            ax.set_title(col)
            ax.legend()
        plt.suptitle("Phase 2 — hierarchy-aware loss at d=8, c=1")
        plt.tight_layout()
        plt.show()

## 7. Phase 3 — Tree-Variant Ablation

**Question:** do conclusions depend on the choice of ground-truth tree?

**Sweep:** train each geometry against `default`, `chronological`, and `flat` trees with the hierarchy-aware loss enabled ($\lambda=1$), then evaluate against the **default** tree. If conclusions flip with the training tree, the result is fragile. Reproduce with `python scripts/sweep.py --phase 3`.

In [ ]:
if sweep is not None:
    p3 = sweep[(sweep["tree_loss_weight"] == 1.0)
               & (sweep["dim"] == 8)].copy()
    if len(p3) > 0:
        agg = p3.groupby(["tree_hierarchy", "geometry"])[
            ["top1_accuracy", "class_center_tree_spearman",
             "tree_distortion_average", "knn_siblings_recall_at_5"]
        ].agg(["mean", "std"]).round(3)
        agg

## 8. Qualitative Analysis

Three lenses on the best models from Phases 1–2:
1. **Poincaré-disk plot of prototypes** (only meaningful at d=2; we run a separate d=2 reference).
2. **Nearest-neighbor grids** for query images, Euclidean vs hyperbolic.
3. **Confusion matrix block-ordered by tree** so off-diagonal mass is interpretable as "how far off in the tree is the typical mistake?"

In [ ]:
from analysis import plot_poincare_disk, plot_confusion_block_tree, plot_knn_image_grid, load_sweep

def _ckpt_for(geometry, dim, **filters):
    """Find the best (highest top-1) checkpoint matching the filters."""
    if not SWEEP_CSV.exists():
        return None
    df = load_sweep(SWEEP_CSV)
    df = df[(df["geometry"] == geometry) & (df["dim"] == dim)]
    for k, v in filters.items():
        df = df[df[k] == v]
    if df.empty:
        return None
    best = df.loc[df["top1_accuracy"].idxmax()]
    return RUNS_DIR / "sweep" / best["config_hash"] / "ckpt.pt"

# 1. Poincaré disk at d=2 — both geometries side by side.
for geo in ["euclidean", "hyperbolic"]:
    ckpt = _ckpt_for(geo, dim=2, tree_loss_weight=0.0, tree_hierarchy="default")
    if ckpt is not None and ckpt.exists():
        fig = plot_poincare_disk(ckpt)
        plt.show()
    else:
        print(f"d=2 {geo} checkpoint not yet available; run sweep --phase 1.")

# 2. Confusion matrix block-ordered by the tree, for the headline winners.
for geo in ["euclidean", "hyperbolic"]:
    ckpt = _ckpt_for(geo, dim=8, tree_loss_weight=0.0, tree_hierarchy="default")
    if ckpt is not None and ckpt.exists():
        fig = plot_confusion_block_tree(ckpt)
        plt.show()
    else:
        print(f"d=8 {geo} checkpoint not yet available.")

# 3. kNN image grid — same query images for both geometries side-by-side.
# We pin a seed so both models get the same queries.
for geo in ["euclidean", "hyperbolic"]:
    ckpt = _ckpt_for(geo, dim=8, tree_loss_weight=0.0, tree_hierarchy="default")
    if ckpt is not None and ckpt.exists() and (DATA_DIR / "wikiart").exists():
        fig = plot_knn_image_grid(ckpt, n_queries=4, k=5, seed=42)
        plt.show()
    elif ckpt is not None and ckpt.exists():
        print("data/wikiart/ not found — skipping kNN grid (needs raw images).")
    else:
        print(f"d=8 {geo} checkpoint not yet available; skipping kNN grid.")

## 9. Headline Results

Final summary table: best Euclidean vs best Hyperbolic across all phases, with seed standard deviations.

In [ ]:
if sweep is not None:
    # "Best" = config with highest top1 across seeds, then report all metrics for that config.
    headline_metrics = ["top1_accuracy", "top5_accuracy", "balanced_accuracy",
                        "class_center_tree_spearman", "tree_distortion_average",
                        "dendrogram_cluster_f1", "knn_siblings_recall_at_5",
                        "knn_cousins_recall_at_5",
                        "frechet_nearest_prototype_accuracy"]
    rows = []
    for geo in ["euclidean", "hyperbolic"]:
        sub = sweep[sweep["geometry"] == geo]
        if len(sub) == 0:
            continue
        # mean over seeds for each (dim, c, lambda) cluster
        cluster_keys = ["dim", "curvature", "tree_loss_weight", "tree_hierarchy"]
        grouped = sub.groupby(cluster_keys, dropna=False)[headline_metrics].agg(["mean", "std"])
        # pick cluster with highest mean top1
        best_idx = grouped[("top1_accuracy", "mean")].idxmax()
        best = grouped.loc[best_idx]
        rows.append((geo, best_idx, best))
    if rows:
        for geo, idx, best in rows:
            print(f"\n=== Best {geo} ===")
            print(f"config: {dict(zip(['dim','curvature','tree_loss_weight','tree_hierarchy'], idx))}")
            print(best.unstack(0).round(4))

## 10. Conclusions

*Filled in after sweeps complete — see the report for the final argument. Expected outcomes:*

1. **Scaling helps both geometries**, but the gap on global metrics is at most marginally closed by raising $d$ alone (Phase 1).
2. **Curvature matters**: at small $c$, the Poincaré ball behaves nearly Euclidean (small region), so its hierarchy advantage — if any — should appear at moderate $c$ where the boundary geometry is felt.
3. **Hierarchy-aware loss** is where geometry should pay off: trees embed isometrically into hyperbolic space but not into Euclidean. If hyperbolic catches up only with $\lambda > 0$, the result is consistent with theory.
4. **Tree variants** (Phase 3) are a sanity check, not a primary result: if the geometry winner flips when we change the training tree, the conclusion is fragile.

## References

- Tan, W. R., Chan, C. S., Aguirre, H. E., & Tanaka, K. (2019). *Improved ArtGAN for conditional synthesis of natural image and artwork.* IEEE Transactions on Image Processing.
- Nickel, M., & Kiela, D. (2017). *Poincaré embeddings for learning hierarchical representations.* NeurIPS.
- Ganea, O., Bécigneul, G., & Hofmann, T. (2018). *Hyperbolic neural networks.* NeurIPS.
- Khrulkov, V., Mirvakhabova, L., Ustinova, E., Oseledets, I., & Lempitsky, V. (2020). *Hyperbolic image embeddings.* CVPR.
- Radford, A., et al. (2021). *Learning transferable visual models from natural language supervision (CLIP).* ICML.
- Chami, I., Gu, A., Chatziafratis, V., & Ré, C. (2020). *From trees to continuous embeddings and back: hyperbolic hierarchical clustering.* NeurIPS.

**Generative AI use.** Claude (Opus 4.7) was used to scaffold the sweep runner and the hierarchy-aware loss implementation, and to help refactor `eval.py` for callable use. All design decisions, model choices, and analysis are the authors'.